# Техническое задание
## Цель:
Разработать аналитический отчёт по пролонгации клиентских проектов за 2023 год
## Задачи:
1. Выполнить загрузку и предварительную обработку данных из файлов prolongations.csv и financial_data.csv;
2. Провести очистку данных;
3. Объединить данные о проектах и финансовых показателях по идентификатору проекта (id);
4. Рассчитать показатели пролонгации для каждого отчётного месяца 2023 года:
сумму проектов, подлежащих пролонгации;
сумму фактически пролонгированных проектов;
коэффициент пролонгации первого месяца;
коэффициент пролонгации второго месяца;
5. Сформировать итоговые аналитические таблицы:
- показатели отдела по месяцам;
- показатели менеджеров по месяцам;
- агрегированные показатели менеджеров за год;
6. Сформировать итоговый Excel-отчёт в соответствии с предоставленным шаблоном оформления.

## Импорт библиотек

In [191]:
import pandas as pd
import numpy as np
import os
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

## Загрузка файлов

In [192]:
try:
    prolongations = pd.read_csv('prolongations.csv')
    financial_data = pd.read_csv('financial_data.csv')
    print("Данные успешно загружены!")
except Exception as e:
    print("Ошибка при загрузке файлов: ", e)

Данные успешно загружены!


## 2. Подготовка данных
### 2.1 Дубликаты, фильтрация

In [193]:
financial_data.columns = financial_data.columns.str.lower().str.strip()
prolongations.columns = prolongations.columns.str.lower().str.strip()

# Приводим к общему регистру строковые поля
prolongations['month'] = prolongations['month'].astype(str).str.lower().str.strip()
prolongations['am'] = prolongations['am'].astype(str).str.strip()

# Гарантируем тип ID (int)
prolongations['id'] = prolongations['id'].astype(int)
financial_data['id'] = financial_data['id'].astype(int)

# Удаление ненужного столбца, если он есть
if 'account' in financial_data.columns:
    financial_data = financial_data.drop(columns=['account'])

# Удаление дубликатов
prolongations = prolongations.drop_duplicates(subset=['id'], keep='first')

# Список колонок-месяцев (все, кроме служебных)
service_cols = ['id', 'причина дубля']
month_columns = [col for col in financial_data.columns if col not in service_cols]


### 2.2 Очистка финансовых данных

In [194]:
# Маска для исключения 'стоп'/'end'
stop_mask = financial_data[month_columns].astype(str).apply(
    lambda x: x.str.strip().str.lower().isin(['стоп', 'end'])
).any(axis=1)
dropped_ids = financial_data[stop_mask]['id'].unique()
financial_data = financial_data[~financial_data['id'].isin(dropped_ids)]

# Очистка числовых значений
for col in month_columns:
    financial_data[col] = (financial_data[col]
                           .astype(str)
                           .str.strip()
                           .str.lower()
                           .replace('в ноль', np.nan)
                           .str.replace(r'\s+', '', regex=True)
                           .str.replace(',', '.', regex=False))
    financial_data[col] = pd.to_numeric(financial_data[col], errors='coerce').fillna(0.0)

# Агрегация дубликатов в финансовых данных по id
financial_data_clean = financial_data.groupby('id')[month_columns].sum().reset_index()


### 2.3. Объединение таблиц и выводные значения

In [195]:
df_merged = pd.merge(prolongations, financial_data_clean, on='id', how='inner')

print(f"Подготовка завершена.")
print(f"Исключено проектов (стоп/end): {len(dropped_ids)}")
print(f"Итого строк в df_merged: {len(df_merged)}")
print(df_merged.head())

Подготовка завершена.
Исключено проектов (стоп/end): 55
Итого строк в df_merged: 258
    id        month                             am  ноябрь 2022  декабрь 2022  \
0   42  ноябрь 2022   Васильев Артем Александрович      36220.0           0.0   
1  453  ноябрь 2022   Васильев Артем Александрович          0.0       39245.0   
2  548  ноябрь 2022      Михайлов Андрей Сергеевич     674000.0      674000.0   
3   87  ноябрь 2022  Соколова Анастасия Викторовна      70050.0           0.0   
4  429  ноябрь 2022  Соколова Анастасия Викторовна      30280.0       35580.0   

   январь 2023  февраль 2023  март 2023  апрель 2023  май 2023  июнь 2023  \
0          0.0           0.0        0.0          0.0       0.0        0.0   
1      44320.0      177635.0        0.0          0.0       0.0        0.0   
2     674000.0      674000.0        0.0          0.0       0.0        0.0   
3      73380.0       83480.0    89300.0      89300.0   78605.0    72485.0   
4      35830.0       42830.0        0.0    

## 3. Формирование базы проекта

In [196]:
base_projects = []

for idx, row in df_merged.iterrows():
    project_id = row['id']
    manager = row['am']
    end_month = row['month']

    if end_month not in month_columns:
        continue

    end_month_idx = month_columns.index(end_month)
    last_shipment = row[end_month]

    # Правило "в ноль": если отгрузка нулевая (из-за изначального текста), берем предыдущий месяц
    if last_shipment == 0.0 and end_month_idx > 0:
        prev_month = month_columns[end_month_idx - 1]
        last_shipment = row[prev_month]

    base_projects.append({
        'id': project_id,
        'Manager': manager,
        'End_Month': end_month,
        'Last_Shipment': last_shipment,
        'end_month_idx': end_month_idx
    })

df_base = pd.DataFrame(base_projects)
print(f"Таблица сформирована. Всего завершенных проектов: {len(df_base)}")
if not df_base.empty:
    print(df_base.head())

Таблица сформирована. Всего завершенных проектов: 258
    id                        Manager    End_Month  Last_Shipment  \
0   42   Васильев Артем Александрович  ноябрь 2022        36220.0   
1  453   Васильев Артем Александрович  ноябрь 2022            0.0   
2  548      Михайлов Андрей Сергеевич  ноябрь 2022       674000.0   
3   87  Соколова Анастасия Викторовна  ноябрь 2022        70050.0   
4  429  Соколова Анастасия Викторовна  ноябрь 2022        30280.0   

   end_month_idx  
0              0  
1              0  
2              0  
3              0  
4              0  


## 4. Расчёт пролонгаций

In [197]:
p1_months = []
p1_shipments = []
is_p1_list = []

p2_months = []
p2_shipments = []
is_p2_list = []

# Индексируем объединенный датасет по id для мгновенного поиска сумм отгрузок
df_merged_indexed = df_merged.set_index('id')
for idx, row in df_base.iterrows():
    p_id = row['id']
    em_idx = row['end_month_idx']

    # Достаем строку с отгрузками по конкретному проекту из df_merged
    merged_row = df_merged_indexed.loc[p_id]
    # 1. Проверяем 1-й месяц пролонгации (индекс + 1)
    if em_idx + 1 < len(month_columns):
        p1_m = month_columns[em_idx + 1]
        p1_s = merged_row[p1_m]
    else:
        p1_m = None
        p1_s = 0.0

    # 2. Проверяем 2-й месяц пролонгации (индекс + 2)
    if em_idx + 2 < len(month_columns):
        p2_m = month_columns[em_idx + 2]
        p2_s = merged_row[p2_m]
    else:
        p2_m = None
        p2_s = 0.0
        # --- ЛОГИКА ТЗ ---
    # Пролонгация 1-го месяца фиксируется, если в следующий месяц отгрузка > 0
    is_p1 = 1 if p1_s > 0 else 0

    # Пролонгация 2-го месяца фиксируется, если в 1-й месяц была тишина (is_p1 == 0),
    # но во 2-й месяц клиент вернулся и совершил отгрузку (> 0)
    is_p2 = 1 if (is_p1 == 0 and p2_s > 0) else 0

    # Сохраняем результаты в списки
    p1_months.append(p1_m)
    p1_shipments.append(p1_s)
    is_p1_list.append(is_p1)

    p2_months.append(p2_m)
    p2_shipments.append(p2_s)
    is_p2_list.append(is_p2)

# Обогащаем нашу базовую таблицу новыми расчетными колонками
df_base['P1_Month'] = p1_months
df_base['P1_Shipment'] = p1_shipments
df_base['Is_P1'] = is_p1_list

df_base['P2_Month'] = p2_months
df_base['P2_Shipment'] = p2_shipments
df_base['Is_P2'] = is_p2_list

print(f"Расчет флагов пролонгаций завершен. Всего обработано строк: {len(df_base)}")

# Отобразим ключевые колонки для проверки корректности связей
preview_cols = ['id', 'End_Month', 'Last_Shipment', 'P1_Month', 'P1_Shipment', 'Is_P1', 'P2_Month', 'P2_Shipment', 'Is_P2']
df_base[preview_cols].head(10)

Расчет флагов пролонгаций завершен. Всего обработано строк: 258


,id,End_Month,Last_Shipment,P1_Month,P1_Shipment,Is_P1,P2_Month,P2_Shipment,Is_P2
0,42,ноябрь 2022,36220.0,декабрь 2022,0.0,0,январь 2023,0.0,0
1,453,ноябрь 2022,0.0,декабрь 2022,39245.0,1,январь 2023,44320.0,0
2,548,ноябрь 2022,674000.0,декабрь 2022,674000.0,1,январь 2023,674000.0,0
3,87,ноябрь 2022,70050.0,декабрь 2022,0.0,0,январь 2023,73380.0,1
4,429,ноябрь 2022,30280.0,декабрь 2022,35580.0,1,январь 2023,35830.0,0
5,600,ноябрь 2022,281417.0,декабрь 2022,175100.0,1,январь 2023,267220.0,0
6,665,ноябрь 2022,10000.0,декабрь 2022,0.0,0,январь 2023,0.0,0
7,586,ноябрь 2022,52240.0,декабрь 2022,0.0,0,январь 2023,0.0,0
8,637,ноябрь 2022,38045.0,декабрь 2022,0.0,0,январь 2023,0.0,0
9,419,ноябрь 2022,0.0,декабрь 2022,0.0,0,январь 2023,0.0,0


## 5. Список отчётных месяцев

In [198]:
report_months_2023 = [
    'январь 2023',
    'февраль 2023',
    'март 2023',
    'апрель 2023',
    'май 2023',
    'июнь 2023',
    'июль 2023',
    'август 2023',
    'сентябрь 2023',
    'октябрь 2023',
    'ноябрь 2023',
    'декабрь 2023'
]

report_months_2023 = [
    m for m in report_months_2023
    if m in month_columns
]



## 6. Функция calculate_metrics

In [199]:
def calculate_metrics(df_slice, target_month):

    target_month = str(target_month).strip().lower()

    month_lookup = {m.strip().lower(): m for m in month_columns}

    if target_month not in month_lookup:
        return 0, 0, 0, 0, 0, 0

    real_month = month_lookup[target_month]

    tgt_idx = month_columns.index(real_month)

    c1_end_month = month_columns[tgt_idx - 1] if tgt_idx - 1 >= 0 else None
    c2_end_month = month_columns[tgt_idx - 2] if tgt_idx - 2 >= 0 else None

    # -------- K1 --------

    denom_1 = 0
    num_1 = 0
    k1 = 0

    if c1_end_month:

        df_c1 = df_slice[
            df_slice['End_Month'] == c1_end_month
        ]

        denom_1 = df_c1['Last_Shipment'].sum()
        num_1 = df_c1['P1_Shipment'].sum()

        if denom_1 > 0:
            k1 = num_1 / denom_1

    # -------- K2 --------

    denom_2 = 0
    num_2 = 0
    k2 = 0

    if c2_end_month:

        df_c2 = df_slice[
            df_slice['End_Month'] == c2_end_month
        ]

        denom_2 = df_c2['Last_Shipment'].sum()
        num_2 = df_c2['P2_Shipment'].sum()

        if denom_2 > 0:
            k2 = num_2 / denom_2

    return denom_1, num_1, k1, denom_2, num_2, k2

## 7. Расчёт всех метрик

In [200]:
unique_managers = df_base['Manager'].unique()
final_rows = []
for manager in unique_managers:

    df_man = df_base[df_base['Manager'] == manager]

    for m in report_months_2023:

        kp1, pr1, k1, kp2, pr2, k2 = calculate_metrics(df_man, m)

        final_rows.append({
            'Type': 'Manager',
            'Name': manager,
            'Month': m,
            'KP1': kp1,
            'PR1': pr1,
            'K1': k1,
            'KP2': kp2,
            'PR2': pr2,
            'K2': k2
        })

for m in report_months_2023:

    kp1, pr1, k1, kp2, pr2, k2 = calculate_metrics(df_base, m)

    final_rows.append({
        'Type': 'Total',
        'Name': 'Всего по отделу',
        'Month': m,
        'KP1': kp1,
        'PR1': pr1,
        'K1': k1,
        'KP2': kp2,
        'PR2': pr2,
        'K2': k2
    })

## 8. Создание df_report

In [201]:
df_report = pd.DataFrame(final_rows)

print(df_report.head())

      Type                          Name         Month         KP1        PR1  \
0  Manager  Васильев Артем Александрович   январь 2023  1693097.68  941761.60   
1  Manager  Васильев Артем Александрович  февраль 2023    34850.00       0.00   
2  Manager  Васильев Артем Александрович     март 2023   458584.42  292584.55   
3  Manager  Васильев Артем Александрович   апрель 2023   534440.90   58240.00   
4  Manager  Васильев Артем Александрович      май 2023   628358.50  121800.00   

         K1         KP2         PR2        K2  
0  0.556236   582479.00   351740.00  0.603867  
1  0.000000  1693097.68  1144844.69  0.676183  
2  0.638017    34850.00    67774.08  1.944737  
3  0.108974   458584.42   242620.00  0.529063  
4  0.193838   534440.90        0.00  0.000000  


9. Формирование листов "Весь отдел", "Менеджеры за год"

In [202]:
df_report = pd.DataFrame(final_rows)

df_department = (
    df_report[df_report['Type'] == 'Total']
    [
        ['Month', 'KP1', 'PR1', 'K1', 'KP2', 'PR2', 'K2']
    ]
    .copy()
)

df_managers_year = (
    df_report[df_report['Type'] == 'Manager']
    .groupby('Name', as_index=False)
    .agg({
        'KP1': 'sum',
        'PR1': 'sum',
        'KP2': 'sum',
        'PR2': 'sum'
    })
)

df_managers_year['K1'] = (
    df_managers_year['PR1'] /
    df_managers_year['KP1']
).fillna(0)

df_managers_year['K2'] = (
    df_managers_year['PR2'] /
    df_managers_year['KP2']
).fillna(0)

df_managers_year = df_managers_year[
    ['Name', 'KP1', 'PR1', 'K1', 'KP2', 'PR2', 'K2']
]

print(f"Расчет окончен. Сгенерировано {len(df_report)} аналитических метрик.")
print(df_department.head())

print(df_managers_year.head())

Расчет окончен. Сгенерировано 132 аналитических метрик.
            Month         KP1         PR1        K1         KP2         PR2  \
120   январь 2023  5918646.86  2625551.61  0.443607  1856379.00  1510430.00   
121  февраль 2023   726045.50   137155.00  0.188907  5918646.86  2872230.03   
122     март 2023   851161.02   498867.65  0.586103   726045.50   217194.08   
123   апрель 2023  1628308.00   445930.35  0.273861   851161.02   453707.00   
124      май 2023  1327728.50   567749.75  0.427610  1628308.00   394321.14   

           K2  
120  0.813643  
121  0.485285  
122  0.299147  
123  0.533045  
124  0.242166  
                           Name         KP1         PR1        K1         KP2  \
0  Васильев Артем Александрович  6093726.60  2431528.15  0.399022  6632105.60   
1       Иванова Мария Сергеевна  3476255.41   833975.80  0.239906  3528495.41   
2      Кузнецов Михаил Иванович   577782.53   222735.00  0.385500   217886.47   
3     Михайлов Андрей Сергеевич  1313289.09   956

In [203]:
print(df_managers_year.columns)

Index(['Name', 'KP1', 'PR1', 'K1', 'KP2', 'PR2', 'K2'], dtype='object')


## 10. Подготовка таблиц для листа "Менеджеры по месяцам"

In [204]:
# Берем помесячные данные менеджеров
df_managers = (
    df_report[df_report['Type'] == 'Manager']
    .copy()
)

print(f"Уникальных записей по менеджерам: {len(df_managers)}")

print("\nПример плоской таблицы (первые 5 строк):")
print(df_managers.head(5))

print("-" * 50)

# Сводная таблица K1
pivot_k1 = df_managers.pivot_table(
    index='Name',
    columns='Month',
    values='K1',
    aggfunc='first'
)

pivot_k1 = pivot_k1.reindex(columns=report_months_2023)

print("\nСводная таблица коэффициентов K1 (1-й месяц) по менеджерам:")
print(pivot_k1.round(3))

print("-" * 50)

# Сводная таблица K2
pivot_k2 = df_managers.pivot_table(
    index='Name',
    columns='Month',
    values='K2',
    aggfunc='first'
)

pivot_k2 = pivot_k2.reindex(columns=report_months_2023)

print("\nСводная таблица коэффициентов K2 (2-й месяц) по менеджерам:")
print(pivot_k2.round(3))

Уникальных записей по менеджерам: 120

Пример плоской таблицы (первые 5 строк):
      Type                          Name         Month         KP1        PR1  \
0  Manager  Васильев Артем Александрович   январь 2023  1693097.68  941761.60   
1  Manager  Васильев Артем Александрович  февраль 2023    34850.00       0.00   
2  Manager  Васильев Артем Александрович     март 2023   458584.42  292584.55   
3  Manager  Васильев Артем Александрович   апрель 2023   534440.90   58240.00   
4  Manager  Васильев Артем Александрович      май 2023   628358.50  121800.00   

         K1         KP2         PR2        K2  
0  0.556236   582479.00   351740.00  0.603867  
1  0.000000  1693097.68  1144844.69  0.676183  
2  0.638017    34850.00    67774.08  1.944737  
3  0.108974   458584.42   242620.00  0.529063  
4  0.193838   534440.90        0.00  0.000000  
--------------------------------------------------

Сводная таблица коэффициентов K1 (1-й месяц) по менеджерам:
Month                          ян

## 11. Выгрузка в Excel

In [205]:
pivot_k1 = pivot_k1.fillna(0)
pivot_k2 = pivot_k2.fillna(0)

output_file = "Отчет_по_пролонгациям.xlsx"

if os.path.exists(output_file):
    os.remove(output_file)

# Запись данных
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    # Весь отдел
    df_department.to_excel(
        writer,
        sheet_name="Весь отдел",
        startrow=2,
        index=False,
        header=False
    )

    # Менеджеры за год
    df_managers_year.to_excel(
        writer,
        sheet_name="Менеджеры за год",
        startrow=2,
        index=False,
        header=False
    )

    # Менеджеры по месяцам
    pivot_k1.to_excel(
        writer,
        sheet_name="Менеджеры по месяцам",
        startrow=1,
        index=False
    )

    start_row = len(pivot_k1) + 5

    pivot_k2.to_excel(
        writer,
        sheet_name="Менеджеры по месяцам",
        startrow=start_row,
        index=False
    )

# Оформление
wb = load_workbook(output_file)

header_fill = PatternFill(
    fill_type="solid",
    fgColor="EDE2BC"
)

thin = Side(style="thin", color="000000")

# границы и выравнивание
for ws in wb.worksheets:
    for row in ws.iter_rows():
        for cell in row:
            cell.border = Border(
                left=thin,
                right=thin,
                top=thin,
                bottom=thin
            )
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True
            )

# Лист Весь отдел
ws = wb["Весь отдел"]

# если вдруг строка с КП1 уже попала
if ws["A3"].value == "Месяц":
    ws.delete_rows(3)

ws.merge_cells("A1:A2")
ws.merge_cells("B1:D1")
ws.merge_cells("E1:G1")

ws["A1"] = "Месяц"

ws["B1"] = "Пролонгации в первый месяц"
ws["E1"] = "Пролонгации через месяц"

ws["B2"] = "к пролонгации"
ws["C2"] = "пролонгировано"
ws["D2"] = "Коэффициент"

ws["E2"] = "к пролонгации"
ws["F2"] = "пролонгировано"
ws["G2"] = "Коэффициент"

for row in ws.iter_rows(min_row=1, max_row=2):
    for cell in row:
        cell.fill = header_fill
        cell.font = Font(bold=True)

# Лист Менеджеры за год
ws = wb["Менеджеры за год"]

# удаляем строку КП1 ПР1 К1 если вдруг появилась
if ws["A3"].value == "Менеджер":
    ws.delete_rows(3)

ws.merge_cells("A1:A2")
ws.merge_cells("B1:D1")
ws.merge_cells("E1:G1")

ws["A1"] = "Менеджер"

ws["B1"] = "Пролонгации в первый месяц"
ws["E1"] = "Пролонгации через месяц"

ws["B2"] = "к пролонгации"
ws["C2"] = "пролонгировано"
ws["D2"] = "Коэффициент"

ws["E2"] = "к пролонгации"
ws["F2"] = "пролонгировано"
ws["G2"] = "Коэффициент"

for row in ws.iter_rows(min_row=1, max_row=2):
    for cell in row:
        cell.fill = header_fill
        cell.font = Font(bold=True)

# Лист Менеджеры по месяцам
ws = wb["Менеджеры по месяцам"]

ws["A1"] = "Коэффициент 1"

row2 = len(pivot_k1) + 5

ws.cell(
    row=row2,
    column=1
).value = "Коэффициент 2"


# Оформление листа "Менеджеры по месяцам"
ws = wb["Менеджеры по месяцам"]

# Более насыщенный желтый для заголовков Коэффициент 1 / Коэффициент 2
section_fill = PatternFill(
    fill_type="solid",
    fgColor="FFD966"
)

# Такой же цвет как на первых двух листах
table_header_fill = PatternFill(
    fill_type="solid",
    fgColor="EDE2BC"
)

# Коэффициент 1
ws["A1"].fill = section_fill
ws["A1"].font = Font(bold=True)

# Заголовок таблицы K1 (строка 2)
for cell in ws[2]:
    cell.fill = table_header_fill
    cell.font = Font(bold=True)

# Коэффициент 2
ws.cell(row=row2, column=1).fill = section_fill
ws.cell(row=row2, column=1).font = Font(bold=True)

# Заголовок таблицы K2
header_row_k2 = row2 + 1

for cell in ws[header_row_k2]:
    cell.fill = table_header_fill
    cell.font = Font(bold=True)

# Немного увеличим высоту строк
ws.row_dimensions[1].height = 25
ws.row_dimensions[2].height = 25

ws.row_dimensions[row2].height = 25
ws.row_dimensions[header_row_k2].height = 25

# ширина столбцов
for ws in wb.worksheets:
    for column in ws.columns:
        try:
            max_length = max(
                len(str(cell.value))
                if cell.value is not None else 0
                for cell in column
            )

            ws.column_dimensions[
                column[0].column_letter
            ].width = max_length + 3

        except:
            pass

wb.save(output_file)

print(f"Отчет сохранен: {output_file}")

Отчет сохранен: Отчет_по_пролонгациям.xlsx
